In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

df = pd.read_csv("../data/processed/traffic_hourly_clean.csv", parse_dates=["date_time"])

df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek


df["pred_naive"] = df["traffic_volume"].shift(168)


df_new = df.dropna(subset=["traffic_volume"]).copy()


train_end = "2016-09-30 23:00"
valid_end = "2017-09-30 23:00"
train = df_new[df_new["date_time"] <= train_end]
validate = df_new[(df_new["date_time"] > train_end) & (df_new["date_time"] <= valid_end)]
test = df_new[df_new["date_time"] > valid_end]

print(len(train), len(validate), len(test))   

23125 8683 8733


In [2]:
avg_table = (
        train.groupby(["day_of_week", "hour"])["traffic_volume"]
            .mean()
                .reset_index()
                    .rename(columns={"traffic_volume": "pred_avg"})
)

In [3]:
validate = validate.drop(columns="pred_avg", errors="ignore").merge(
        avg_table, on=["day_of_week", "hour"], how="left"
        )


In [4]:
validate["pred_const"] = train["traffic_volume"].mean()

In [5]:
cmp = validate.dropna(subset=["pred_naive", "pred_avg"])
print("rows compared:", len(cmp))

y = cmp["traffic_volume"]
mae_const = mean_absolute_error(y, cmp["pred_const"])
mae_naive = mean_absolute_error(y, cmp["pred_naive"])
mae_avg = mean_absolute_error(y, cmp["pred_avg"])

mean_traffic = y.mean()
print(f"Mean traffic (validation): {mean_traffic:.0f}")
print(f"Constant  MAE: {mae_const:.0f}  ({mae_const / mean_traffic:.1%} of mean)")
print(f"Naive-168 MAE: {mae_naive:.0f}  ({mae_naive / mean_traffic:.1%} of mean)")
print(f"Hour-of-week avg MAE: {mae_avg:.0f}  ({mae_avg / mean_traffic:.1%} of mean)")

rows compared: 8606
Mean traffic (validation): 3317
Constant  MAE: 1719  (51.8% of mean)
Naive-168 MAE: 302  (9.1% of mean)
Hour-of-week avg MAE: 277  (8.3% of mean)


In [6]:

df["lag_1"] = df["traffic_volume"].shift(1)

df["lag_24"] = df["traffic_volume"].shift(24)


df[["date_time", "traffic_volume", "lag_1", "lag_24"]].head(30)

,date_time,traffic_volume,lag_1,lag_24
0,2012-10-02 09:00:00,5545.0,NaN,NaN
1,2012-10-02 10:00:00,4516.0,5545.0,NaN
2,2012-10-02 11:00:00,4767.0,4516.0,NaN
3,2012-10-02 12:00:00,5026.0,4767.0,NaN
4,2012-10-02 13:00:00,4918.0,5026.0,NaN
5,2012-10-02 14:00:00,5181.0,4918.0,NaN
6,2012-10-02 15:00:00,5584.0,5181.0,NaN
7,2012-10-02 16:00:00,6015.0,5584.0,NaN
8,2012-10-02 17:00:00,5791.0,6015.0,NaN
9,2012-10-02 18:00:00,4770.0,5791.0,NaN
